In [1]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from lightgbm import LGBMRegressor
from scipy.stats import spearmanr
from sklearn.metrics import root_mean_squared_error

from src.data import load_fd001
from src.features import build_features
from src.metrics import nasa_score

In [2]:
df = pd.read_pickle("../data/processed/fd001_basis_c.pkl")
feature_cols = [c for c in df.columns if c not in ["unit", "RUL"]]

In [3]:
X = df[feature_cols]
y = df["RUL"]

model = LGBMRegressor(verbose = -1, n_jobs = 1, random_state=42)
model.fit(X, y)

,random_state,42
,n_jobs,1
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None


In [4]:
explainer = shap.TreeExplainer(model)
shap_werte = explainer.shap_values(X)

In [5]:
shap_werte.shape

(20631, 98)

In [6]:
np.allclose(model.predict(X), (shap_werte.sum(axis=1) + explainer.expected_value))

True

In [7]:
print(explainer.expected_value)
print(y.mean())

86.82928602578225
86.82928602588338


In [8]:
feature_shap = np.abs(shap_werte).mean(axis = 0)
feature_importance = pd.Series(feature_shap, index = feature_cols)
feature_importance.sort_values(ascending=False).head(10)

s4_mean5       7.540240
s15_mean5      3.734426
s11_slope25    3.448518
s11_mean5      2.717997
s12_slope25    2.672856
s9_slope25     2.330824
s8_slope25     2.208033
s2_mean5       2.107821
s14_slope25    2.070767
s13_slope25    1.898039
dtype: float64

In [9]:
sensor_namen = []
typen = []

for col in feature_cols:
    teile = col.split("_", 1)
    sensor_namen.append(teile[0])
    if len(teile)==1:
        name = "roh"
        typen.append(name)
    else:
        typen.append(teile[1])


shap_df = pd.DataFrame({
    "feature": feature_cols,
    "wichtigkeit": feature_importance.to_numpy(),
    "sensor": sensor_namen,
    "typ": typen,
})


shap_df

,feature,wichtigkeit,sensor,typ
0,s2,0.000000,s2,roh
1,s3,0.004061,s3,roh
2,s4,0.001866,s4,roh
3,s8,0.000000,s8,roh
4,s11,0.000000,s11,roh
...,...,...,...,...
93,s12_slope25,2.672856,s12,slope25
94,s20_slope25,1.021076,s20,slope25
95,s21_slope25,1.063290,s21,slope25
96,s9_slope25,2.330824,s9,slope25


In [10]:
shap_df.groupby("typ")["wichtigkeit"].sum().sort_values(ascending=False)

typ
slope25     23.146532
mean5       22.291218
mean25       7.673696
std25        3.144533
diff5_25     0.679464
std5         0.288214
roh          0.094265
Name: wichtigkeit, dtype: float64

In [11]:
shap_df.groupby("sensor")["wichtigkeit"].sum().sort_values(ascending = False)

sensor
s4     9.440385
s11    6.867632
s15    5.456796
s9     5.172837
s14    4.273642
s2     3.872677
s12    3.475552
s8     3.129451
s21    2.750167
s17    2.710189
s7     2.703006
s20    2.601898
s13    2.547243
s3     2.316447
Name: wichtigkeit, dtype: float64

In [12]:
shap.summary_plot(shap_werte, X, max_display=15, show = False)
plt.savefig("../reports/figures/beeswarm_top10_features.png")
plt.close()

In [13]:
typ_werte = shap_df.groupby("typ")["wichtigkeit"].sum().sort_values(ascending=True)

plt.barh(typ_werte.index, typ_werte.values)
plt.xlabel("Summe der mittleren |SHAP| (Zyklen)")
plt.ylabel("Wichtigkeit je Feature-Typ (LGBM Default, Trainingsdaten)")
plt.savefig("../reports/figures/wichtigkeit_nach_typ.png")
plt.close()

Hypothese SHAP Teil 2 (vorab, 30.10.)
s11_slope25: fallend (rechts unten), Spearman ρ ohne die künstlichen Nullen < −0,65 · flacher Bereich links (Sichtprüfung) · bei x = 0 genau 2.400 Zeilen, mittlerer SHAP > 0, Knick der Funktion bei x = 0,003
s9_slope25: gleiche Richtung wie s11, mehr senkrechte Streuung → |ρ| um mindestens 0,35 kleiner als bei s11 (bezogen auf den tatsächlichen s11-Wert)

In [14]:
n_nullen = (X["s11_slope25"]==0).sum()

In [15]:
spalte_s11_slope = X.columns.get_loc("s11_slope25")
shap_s11_slope = shap_werte[:,spalte_s11_slope]

In [16]:
s11_slope = X["s11_slope25"].to_numpy()
s11_slope_mask = s11_slope != 0

In [17]:
x_spear = s11_slope[s11_slope_mask]
y_spear = shap_s11_slope[s11_slope_mask]

print(len(y_spear))

spear_s11_slope = spearmanr(x_spear, y_spear).statistic
shap_s11_slope_x0 = shap_s11_slope[~s11_slope_mask].mean()

18231


In [18]:
shap.dependence_plot("s11_slope25", shap_werte, X, interaction_index=None, show=False)
plt.savefig("../reports/figures/shap_dependence_plot_s11_slope.png")
plt.close()

In [19]:
ergebnisse = []

for col in ["s11_slope25", "s9_slope25"]:
    n_nullen = (X[col] == 0).sum()
    spalte = X.columns.get_loc(col)
    shap_col = shap_werte[:, spalte]
    slope = X[col].to_numpy()
    col_mask = slope != 0

    x_spear = slope[col_mask]
    y_spear = shap_col[col_mask]

    rho = spearmanr(x_spear, y_spear).statistic
    mean_at_0 = shap_col[~col_mask].mean()

    ergebnisse.append({
        "feature": col,
        "n_nullen": n_nullen,
        "rho": rho,
        "shap bei 0":  mean_at_0,
    })

    shap.dependence_plot(col, shap_werte, X, interaction_index=None, show= False)
    plt.savefig(f"../reports/figures/shap_dependence_plot_{col}.png", dpi = 150, bbox_inches = "tight")
    plt.close()


tabelle = pd.DataFrame(ergebnisse)


In [20]:
tabelle

,feature,n_nullen,rho,shap bei 0
0,s11_slope25,2400,-0.912600,3.063493
1,s9_slope25,2400,-0.571067,1.474721


Abgleich von s9 Slope mit EDA auf Triebwerksebene:

Hypothese:
Bei höchstens 70 von 100 Triebwerken ist die `s9_slope25` im letzten Zyklus > 0.

In [21]:
last_units_s9_slope = df[df["RUL"]==0]["s9_slope25"]
assert len(last_units_s9_slope)==df["unit"].nunique(), "Falsche Länge"
np.count_nonzero(last_units_s9_slope > 0)

np.int64(68)

### SHAP Teil 2: Abhängigkeitsplots `s11_slope25` und `s9_slope25`

**Hypothese (vorab, 30.10.)**
- `s11_slope25`: fallend, Spearman ρ ohne die künstlichen Nullen < −0,65 · flacher Bereich links · Knick rechts von 0 bei ca. 0,003 · bei x = 0 genau 2.400 Zeilen (24 Zyklen × 100 Triebwerke wegen `min_periods=25`), mittlerer SHAP > 0
- `s9_slope25`: gleiche Richtung wie s11, mehr senkrechte Streuung → |ρ| um mindestens 0,35 kleiner als bei s11
- Triebwerks-Ebene: bei höchstens 70 von 100 Triebwerken ist `s9_slope25` im letzten Zyklus > 0

Spearman statt Pearson, weil eine monotone, aber nicht lineare Form erwartet wird (Bäume bilden Stufen, die Kappung erzeugt ein Plateau). Die Nullen bleiben draußen, weil sie Platzhalter für die ersten 24 Zyklen sind, keine Messungen.

**Ergebnis**

| Prüfung | Hypothese | Ergebnis | |
|---|---|---|---|
| s11: Zeilen bei x = 0 | 2.400 | 2.400 | ✓ |
| s11: ρ ohne Nullen | < −0,65 | −0,91 | ✓ |
| s11: mittlerer SHAP bei x = 0 | > 0 | +3,06 | ✓ |
| s11: flacher Bereich | links | oberes Plateau ca. +3 bis +5 | ✓ |
| s11: Knick | rechts von 0, ca. 0,003 | ca. 0,002 | ✓ |
| s9: \|ρ\| ohne Nullen | ≤ \|ρ_s11\| − 0,35 | -0,57 | ✗ |
| s9: Streuung | mehr als bei s11 | SHAP von ca. +5 bis −20 | ✓ |
| s9 im letzten Zyklus > 0 | ≤ 70 von 100 | 68 von 100 | ✓ |

→ **s11 bestätigt.** **s9 knapp widerlegt:** Die Vermutung zeigt in die richtige Richtung (|ρ| fällt von 0,91 auf 0,57, die Streuung ist deutlich größer). Die vorab festgelegte Schwelle ist aber verfehlt, der Abstand beträgt 0,3416 statt ≥ 0,35. Die Schwelle wird nicht nachträglich angepasst.

**Deutung**
- **Die Form ist eine Stufe mit zwei Plateaus**, kein linearer Verlauf. Das obere Plateau entsteht durch die Kappung bei 125: Alle Zeilen dort haben dasselbe Label, das Modell hat keinen Grund, sie zu unterscheiden. Das untere Plateau entsteht, weil die RUL nicht unter 0 fallen kann, rechts nur wenige Punkte liegen und Bäume nicht extrapolieren.
- **Der Knick liegt rechts von 0:** s11 steigt schon leicht, bevor das Modell abzieht. Frühe Degradation wird durch die Kappung als „gesund“ gelabelt. Das ist ein Argument für die Kappungs-Sensitivität (115/125/135).
- **Die x-Achsen sind nicht vergleichbar:** s9 misst eine Drehzahl, die Steigungen sind rund 100× größer als bei s11.
- **s9 ist ein einseitiges Signal:** Bei 32 von 100 Triebwerken fällt s9 am Lebensende. Damit ist das EDA-Urteil „uneindeutig“ auf Triebwerks-Ebene beziffert. Ein flaches s9 bringt kaum Bonus (oberes Plateau nur ca. 0 bis +2 statt +3 bis +5 bei s11). Ein steigender s9 ist ein Alarm, ein fallender s9 sagt nichts. Das passt zu SHAP Teil 1, wo s9 sich wie ein steigender Sensor verhielt.
  - *Vermutung, ungeprüft:* Die dünne Linie links (x ≈ −0,6 bis −0,2, SHAP ≈ +0,3) sind die späten Zeilen genau dieser 32 Triebwerke. Das Modell ignoriert s9 dort.

**Offen (Vermutungen, ungeprüft)**
- **Trichter nach der Stufe:** Bei gleichem x liegen die SHAP-Werte zwischen ca. −4 und −16 (s11) bzw. −20 (s9). Das ist Wechselwirkung. Vermutung: Die Steigung teilt sich den Kredit mit `mean5` und bekommt dort großes Minus, wo `mean5` den Verschleiß noch nicht zeigt. Prüfung: `interaction_index="s11_mean5"`.
- **Spitze bei x = 0:** s9 reicht dort bis ca. +4, s11 bis ca. +6,5. Vermutung: Das Modell hat den Platzhalter 0 als Altersmarker gelernt („erste 24 Zyklen = jung“). Das ist kein Leakage, weil die Zykluszahl im Betrieb bekannt ist, aber ein Artefakt der Feature-Konstruktion. Prüfung: mittlerer SHAP bei exakt 0 gegen echte Steigungen nahe 0 (z. B. s9 mit |x| < 0,05 und x ≠ 0). Für den Testlauf irrelevant, wenn das kürzeste Testtriebwerk mindestens 25 Zyklen hat.

Plots: `reports/figures/shap_dependence_plot_s11_slope25.png`, `reports/figures/shap_dependence_plot_s9_slope25.png`

In [22]:
train, test, rul = load_fd001()

In [23]:
rul.describe()

,RUL
count,100.00000
mean,75.52000
std,41.76497
min,7.00000
25%,32.75000
50%,86.00000
75%,112.25000
max,145.00000


In [24]:
print(rul["RUL"].between(40,80).mean())
print((rul["RUL"]>125).mean())
print(df["RUL"].between(40,80).mean())

0.17
0.11
0.19873006640492463


### Testlauf: Hypothese und Regeln (vorab, 30.10.)

**Regel:** Das Testset wird genau einmal ausgewertet. Unabhängig vom Ergebnis bleibt das finale Modell LGBM Default auf Basis c (98 Features). Es gibt keine Anpassung nach dem Blick auf den Test.

**Entscheidungen**
- **Hauptmetrik: RMSE gegen die ungekappte `RUL_FD001`.** Sie misst, was das Modell im realen Einsatz liefert. Die RMSE gegen die bei 125 gekappte Test-RUL wird zusätzlich berichtet, um mit der Literatur vergleichbar zu sein. Gekappt ≤ ungekappt gilt zwingend, weil das Modell nie über 125 vorhersagt. Das ist deshalb keine Hypothese.
- **NASA-Score als Mittelwert** (wie in der CV) **und als Summe** über 100 Triebwerke (wie in der Literatur üblich).
- **Vergleichswert:** CV-RMSE von LGBM Default = 13,24, berechnet auf dem Mac mini, demselben Rechner wie dieser Lauf.

**Vergleichbarkeit CV gegen Test.** Die CV bewertet alle Zeilen, der Test nur den letzten Zyklus von 100 abgeschnittenen Triebwerken.

| Bereich | CV (Zeilen) | Test (Triebwerke) | erwartete Wirkung auf die Test-RMSE |
|---|---|---|---|
| Plateau (RUL ≥ 125) | ca. 42 % | 11 % (RUL > 125) | ↑ weniger leichte Fälle* |
| 40–80 (größter OOF-Fehler) | 19,9 % | 17 % | ≈ fast gleich |
| über 125, ungekappt | 0 (Labels gekappt) | 11 Triebwerke | ↑ eingebauter Fehler, da Vorhersage ≤ 125 |

\* Annahme: Der OOF-Fehler auf dem Plateau liegt unter dem Durchschnitt. [PLATZHALTER: in `04` prüfen]

**Hypothese**
- **RMSE (ungekappt)** liegt über der CV-RMSE von 13,24, im Bereich **13,8 bis 14,6**. Begründung: Im Test fehlen viele Plateau-Fälle, und die 11 Triebwerke mit wahrer RUL über 125 haben einen eingebauten Fehler, den es in der CV nicht gab. Der schwierige Bereich 40–80 ist im Test etwa gleich stark vertreten. Das Band ist bewusst nicht enger: Bei 100 Triebwerken hebt ein einzelnes Triebwerk mit 40 Zyklen Fehler die RMSE schon von 13,24 auf ca. 13,8.
- **Bias < 0 (zu früh).** Begründung: Die 11 Triebwerke über 125 werden zwingend zu früh vorhergesagt. Das überwiegt den „zu späten“ Fehler im mittleren Bereich aus der OOF-Analyse, weil dieser Bereich im Test etwas schwächer vertreten ist.

In [25]:
test_feat, test_cols = build_features(test)
assert list(test_cols) == list(X.columns), "nicht übereinstimmend"

In [26]:
test_feat.groupby("unit")["cycle"].max().min()

np.int64(31)

In [27]:
letzte = test_feat.groupby("unit").tail(1)
assert len(letzte) == test_feat["unit"].nunique(), "nicht übereinstimmend"
assert list(letzte["unit"])==list(range(1,101)), "Reihenfolge der Triebwerke passt nicht"

In [28]:
model.get_params()

{'boosting_type': 'gbdt',
 'class_weight': None,
 'colsample_bytree': 1.0,
 'importance_type': 'split',
 'learning_rate': 0.1,
 'max_depth': -1,
 'min_child_samples': 20,
 'min_child_weight': 0.001,
 'min_split_gain': 0.0,
 'n_estimators': 100,
 'n_jobs': 1,
 'num_leaves': 31,
 'objective': None,
 'random_state': 42,
 'reg_alpha': 0.0,
 'reg_lambda': 0.0,
 'subsample': 1.0,
 'subsample_for_bin': 200000,
 'subsample_freq': 0,
 'verbose': -1}

In [29]:
letzte[feature_cols].shape

(100, 98)

In [30]:
pred = model.predict(letzte[feature_cols])

In [31]:
pred.max()

np.float64(125.28562976565023)

Gekappt ≤ ungekappt gilt praktisch immer, weil die Vorhersagen höchstens knapp über 125 liegen (Maximum 125,28; Boosting summiert Bäume und kann das höchste Label leicht überschreiten). Das ist deshalb keine Hypothese.

In [32]:
y_test = rul["RUL"].to_numpy()
assert y_test.shape == pred.shape, "nicht übereinstimmend"

In [33]:
y_test_capped = np.minimum(y_test, 125)

In [35]:
varianten = [("uncapped", y_test), ("capped", y_test_capped)]
test_ergebnisse = []

for name, y_wahr in varianten:
    
    pred = model.predict(letzte[feature_cols])
    fehler = pred - y_wahr

    test_ergebnisse.append({
        "variante": name,
        "rmse": root_mean_squared_error(y_wahr, pred),
        "nasa_mean": nasa_score(y_wahr, pred,aggregation = "mean"),
        "nasa_sum": nasa_score(y_wahr, pred, aggregation = "sum"),
        "bias": fehler.mean(),
        "anteil_spaet": (fehler>0).mean(),
        "max_spaet": fehler.max(),
    })

test_tabelle = pd.DataFrame(test_ergebnisse)

test_tabelle.to_csv("../reports/ergebnisse/vergleich_aus_06.csv", index = False)

### Testlauf: Ergebnis

| Variante | RMSE | NASA mittel | NASA Summe | Bias | Anteil zu spät | max. zu spät |
|---|---|---|---|---|---|---|
| **ungekappt (Hauptmetrik)** | **13,00** | 2,48 | 248,2 | +1,13 | 60 % | 30,0 |
| gekappt bei 125 | 11,79 | 2,20 | 219,5 | +2,20 | 61 % | 30,0 |
| CV LGBM Default (Mac mini) | 13,24 | 3,60 | – | −0,10 | 46 % | 54,2* |

\* Maximum über 20.631 Zeilen statt 100 Triebwerke, nicht direkt vergleichbar.

**Checks:** Spalten wie im Training ✓ · kürzestes Testtriebwerk 31 Zyklen ≥ 25, der Platzhalter 0 kommt im letzten Zyklus nicht vor ✓ · 100 Zeilen in der Reihenfolge 1–100 ✓ · Modell = Default (`learning_rate` 0,1, `n_estimators` 100) ✓ · Vorhersage-Maximum 125,28 ✓ · gekappt ≤ ungekappt ✓

**Hypothese:** RMSE 13,8–14,6 → **widerlegt** (13,00, unter der CV) · Bias < 0 → **widerlegt** (+1,13, zu spät)

**Warum widerlegt**
- **Bias durch die Zusammensetzung:** Auf dem Plateau (Wahrheit 125, Vorhersage ≤ 125,28) ist das Modell praktisch nie zu spät. In der CV sind 42 % der Zeilen Plateau-Zeilen. Sie gleichen das „zu spät“ im mittleren Bereich aus (Bias −0,10, 46 % zu spät). Im Test sind es nur 11 %, deshalb überwiegt das „zu spät“ (Bias +1,13, 60 % zu spät).
- **Denkfehler beim eingebauten Fehler:** Ein Fehler hebt die RMSE nur, wenn er größer ist als der typische Fehler. Die Bias-Differenz zwischen gekappt und ungekappt ergibt (2,20 − 1,13) × 100 / 11 ≈ 9,7. Die 11 Triebwerke über 125 liegen also im Mittel nur ca. 10 Zyklen darüber (mittlere wahre RUL 134,72). Das ist weniger als der typische Fehler von ca. 13, deshalb senken sie die RMSE eher. Aus demselben Grund ist ihr „zu früh“ zu klein, um viel auszugleichen.
- **Plateau-Annahme ungeprüft:** Ob im Test „leichte“ Plateau-Fälle fehlen, hängt vom OOF-Fehler auf dem Plateau ab. [offen: Plateau-RMSE gegen Gesamt-RMSE in der OOF]
- **Rauschen:** Der Abstand zur CV beträgt −0,24. Ein einzelnes Triebwerk verschiebt die RMSE um ca. 0,5.

**Einordnung**
- Die Test-RMSE liegt im Bereich der CV. Die CV-Schätzung war realistisch, und es gibt keinen Hinweis auf eine Überanpassung der Modellwahl an die CV-Folds.
- 60 % der Vorhersagen sind zu spät, maximal um 30 Zyklen. Das ist die sicherheitsrelevante Seite und die Motivation für `07_decision` (Unsicherheit, Wartungskosten).
- Das Modell bleibt LGBM Default, wie vorab festgelegt.

In [ ]:
y_test[y_test > 125].mean()

np.float64(134.72727272727272)